In [1]:
!pip -q install scikit-learn xgboost scipy

In [2]:
import os
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
from glob import glob
from scipy.stats import skew, kurtosis

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

In [3]:
os.makedirs("data/10_round1_improved_baseline", exist_ok=True)

In [4]:
df = pd.read_csv("dataset_v2_price_plus_dictionary_updated.csv")

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()

Shape: (2247, 23)
['ticker', 'cik', 'filing_date', 'filing_type', 'accession_number', 'year', 'quarter', 'cik_nolead', 'acc_nodash', 'mda_path', 'text_length_words', 'mda_status', 'primary_doc', 'past_return_10d', 'past_realized_vol_5d', 'past_realized_vol_10d', 'future_realized_vol_10d', 'abs_past_return_10d', 'lm_negative', 'lm_positive', 'lm_uncertainty', 'lm_total_words', 'lm_net_sentiment']


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,past_return_10d,past_realized_vol_5d,past_realized_vol_10d,future_realized_vol_10d,abs_past_return_10d,lm_negative,lm_positive,lm_uncertainty,lm_total_words,lm_net_sentiment
0,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1,320193,32019319000010,AAPL_20190130_10-Q_000032019319000010.txt,...,0.031200,0.018383,0.016375,0.012985,0.031200,0.040979,0.005424,0.034711,8297,-0.035555
1,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,0.007228,0.008079,0.010983,0.022134,0.007228,0.040265,0.005329,0.033989,8444,-0.034936
2,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.020931,0.006725,0.011068,0.028255,0.020931,0.040797,0.005400,0.034677,8334,-0.035397
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,0.002303,0.020648,0.015755,0.021234,0.002303,0.005647,0.001694,0.010164,3542,-0.003953
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.024802,0.020703,0.023482,0.012307,0.024802,0.008942,0.002835,0.009378,4585,-0.006107


In [5]:
df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce")
df = df.dropna(subset=["filing_date"]).copy()

df = df.sort_values(["ticker", "filing_date"]).reset_index(drop=True)

print("Shape after sorting:", df.shape)

Shape after sorting: (2247, 23)


In [6]:
df["is_10k"] = (df["filing_type"].astype(str).str.upper() == "10-K").astype(int)

df["filing_month"] = df["filing_date"].dt.month.astype("Int64")
df["filing_year"] = df["filing_date"].dt.year.astype("Int64")

df["log_text_length_words"] = np.log(df["text_length_words"].clip(lower=1))

In [8]:
zip_path = Path("ticker_price_files_filtered_updated.zip")
extract_dir = Path("data/04_prices/ticker_price_files_filtered_updated")

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extracted price files to:", extract_dir)
print("Total extracted files:", len(list(extract_dir.glob("*.csv"))))

Extracted price files to: data/04_prices/ticker_price_files_filtered_updated
Total extracted files: 123


In [9]:
price_files = glob(str(extract_dir / "*.csv"))

price_data = {}

for fp in price_files:
    ticker = Path(fp).stem
    px = pd.read_csv(fp)

    px["date"] = pd.to_datetime(px["date"], errors="coerce")

    if "ret" in px.columns and "return" not in px.columns:
        px = px.rename(columns={"ret": "return"})

    px["return"] = pd.to_numeric(px["return"], errors="coerce")
    px["adj_close"] = pd.to_numeric(px["adj_close"], errors="coerce")

    px = px.dropna(subset=["date", "return", "adj_close"]).copy()
    px = px.sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

    price_data[ticker] = px

print("Loaded price files:", len(price_data))
print("Example tickers:", list(price_data.keys())[:10])

Loaded price files: 123
Example tickers: ['WMB', 'FDX', 'CMCSA', 'ISRG', 'EQIX', 'SYK', 'SHW', 'NVDA', 'NEM', 'DHR']


In [10]:
def cumulative_return(returns):
    returns = np.asarray(returns)
    return np.prod(1 + returns) - 1

def realized_volatility(returns):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1)

def safe_skew(x):
    x = np.asarray(x)
    if len(x) < 3:
        return np.nan
    return skew(x, bias=False)

def safe_kurtosis(x):
    x = np.asarray(x)
    if len(x) < 4:
        return np.nan
    return kurtosis(x, fisher=True, bias=False)

def mean_abs_return(x):
    x = np.asarray(x)
    return np.mean(np.abs(x))

def max_abs_return(x):
    x = np.asarray(x)
    return np.max(np.abs(x))

In [11]:
rows = []
dropped = []

for _, row in df.iterrows():
    ticker = str(row["ticker"]).strip().upper()
    filing_date = row["filing_date"]

    if ticker not in price_data:
        dropped.append((ticker, filing_date, "NO_PRICE_FILE"))
        continue

    px = price_data[ticker]

    past = px[px["date"] < filing_date].copy()
    future = px[px["date"] > filing_date].copy()

    if len(past) < 60:
        dropped.append((ticker, filing_date, "NOT_ENOUGH_PAST_60"))
        continue

    if len(future) < 10:
        dropped.append((ticker, filing_date, "NOT_ENOUGH_FUTURE_10"))
        continue

    past_5 = past.tail(5)
    past_10 = past.tail(10)
    past_20 = past.tail(20)
    past_30 = past.tail(30)
    past_60 = past.tail(60)

    future_10 = future.head(10)

    r5 = past_5["return"].values
    r10 = past_10["return"].values
    r20 = past_20["return"].values
    r30 = past_30["return"].values
    r60 = past_60["return"].values
    rfut = future_10["return"].values

    new_row = row.to_dict()

    # original Step 5 features
    new_row["past_return_10d"] = cumulative_return(r10)
    new_row["abs_past_return_10d"] = abs(new_row["past_return_10d"])
    new_row["past_realized_vol_5d"] = realized_volatility(r5)
    new_row["past_realized_vol_10d"] = realized_volatility(r10)
    new_row["future_realized_vol_10d"] = realized_volatility(rfut)

    # new improved baseline features
    new_row["past_return_5d"] = cumulative_return(r5)
    new_row["past_return_20d"] = cumulative_return(r20)

    new_row["past_realized_vol_20d"] = realized_volatility(r20)
    new_row["past_realized_vol_30d"] = realized_volatility(r30)
    new_row["past_realized_vol_60d"] = realized_volatility(r60)

    new_row["mean_abs_return_5d"] = mean_abs_return(r5)
    new_row["mean_abs_return_10d"] = mean_abs_return(r10)
    new_row["max_abs_return_10d"] = max_abs_return(r10)

    new_row["return_skew_10d"] = safe_skew(r10)
    new_row["return_kurtosis_10d"] = safe_kurtosis(r10)

    new_row["log_price_tminus1"] = np.log(float(past_10["adj_close"].iloc[-1]))

    new_row["vol_ratio_5_20"] = new_row["past_realized_vol_5d"] / (new_row["past_realized_vol_20d"] + 1e-8)
    new_row["vol_ratio_10_60"] = new_row["past_realized_vol_10d"] / (new_row["past_realized_vol_60d"] + 1e-8)

    rows.append(new_row)

rich_df = pd.DataFrame(rows)
dropped_df = pd.DataFrame(dropped, columns=["ticker", "filing_date", "reason"])

print("Rich dataset shape:", rich_df.shape)
print("Dropped rows:", len(dropped_df))
print(dropped_df["reason"].value_counts(dropna=False))

Rich dataset shape: (2181, 40)
Dropped rows: 66
reason
NOT_ENOUGH_PAST_60    66
Name: count, dtype: int64


In [12]:
required_cols = [
    "past_return_5d",
    "past_return_10d",
    "past_return_20d",
    "abs_past_return_10d",
    "past_realized_vol_5d",
    "past_realized_vol_10d",
    "past_realized_vol_20d",
    "past_realized_vol_30d",
    "past_realized_vol_60d",
    "mean_abs_return_5d",
    "mean_abs_return_10d",
    "max_abs_return_10d",
    "return_skew_10d",
    "return_kurtosis_10d",
    "log_price_tminus1",
    "vol_ratio_5_20",
    "vol_ratio_10_60",
    "future_realized_vol_10d",
    "log_text_length_words",
    "is_10k",
    "quarter",
    "filing_month",
    "filing_year"
]

rich_df = rich_df.dropna(subset=required_cols).copy()
rich_df["log_future_realized_vol_10d"] = np.log(rich_df["future_realized_vol_10d"].clip(lower=1e-8))

print("Final rich_df shape:", rich_df.shape)
rich_df.head()

Final rich_df shape: (2181, 41)


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,past_realized_vol_60d,mean_abs_return_5d,mean_abs_return_10d,max_abs_return_10d,return_skew_10d,return_kurtosis_10d,log_price_tminus1,vol_ratio_5_20,vol_ratio_10_60,log_future_realized_vol_10d
0,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,0.011273,0.007233,0.007704,0.019473,-0.006040,0.587393,3.915367,0.870016,0.974208,-3.810629
1,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.015601,0.005166,0.008841,0.022854,0.360768,-0.039101,3.954987,0.662464,0.709436,-3.566474
2,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,...,0.015541,0.009446,0.008896,0.023128,-1.456441,3.091214,4.107836,1.159828,0.733736,-4.628291
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,0.011725,0.013792,0.011713,0.029405,-0.158909,0.862831,4.374782,1.468833,1.343708,-3.852132
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.044873,0.019945,0.019764,0.032845,0.076090,-1.717656,4.296605,0.738856,0.523305,-4.397591


In [13]:
rich_df.to_csv("data/10_round1_improved_baseline/round1_improved_baseline_dataset.csv", index=False)
dropped_df.to_csv("data/10_round1_improved_baseline/round1_improved_baseline_dropped_rows.csv", index=False)

print("Saved round 1 dataset and drop log.")

Saved round 1 dataset and drop log.


In [14]:
split_date = pd.Timestamp("2023-01-01")

train_df = rich_df.loc[rich_df["filing_date"] < split_date].copy().reset_index(drop=True)
test_df = rich_df.loc[rich_df["filing_date"] >= split_date].copy().reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train range:", train_df["filing_date"].min(), "to", train_df["filing_date"].max())
print("Test range:", test_df["filing_date"].min(), "to", test_df["filing_date"].max())

Train rows: 1394
Test rows: 787
Train range: 2019-04-02 00:00:00 to 2022-12-29 00:00:00
Test range: 2023-01-05 00:00:00 to 2024-12-13 00:00:00


In [15]:
price_plus_context_features = [
    "past_return_5d",
    "past_return_10d",
    "past_return_20d",
    "abs_past_return_10d",
    "past_realized_vol_5d",
    "past_realized_vol_10d",
    "past_realized_vol_20d",
    "past_realized_vol_30d",
    "past_realized_vol_60d",
    "mean_abs_return_5d",
    "mean_abs_return_10d",
    "max_abs_return_10d",
    "return_skew_10d",
    "return_kurtosis_10d",
    "log_price_tminus1",
    "vol_ratio_5_20",
    "vol_ratio_10_60",
    "is_10k",
    "log_text_length_words"
]

categorical_features = ["quarter", "filing_month", "filing_year"]

lm_features = [
    "lm_negative",
    "lm_positive",
    "lm_uncertainty",
    "lm_net_sentiment"
]

target_col = "log_future_realized_vol_10d"

feature_sets = {
    "round1_baseline_price_context": price_plus_context_features + categorical_features,
    "round1_plus_lm": price_plus_context_features + lm_features + categorical_features
}

In [16]:
X_train_dict = {name: train_df[cols].copy() for name, cols in feature_sets.items()}
X_test_dict = {name: test_df[cols].copy() for name, cols in feature_sets.items()}

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

for name in feature_sets:
    print(name, X_train_dict[name].shape, X_test_dict[name].shape)

round1_baseline_price_context (1394, 22) (787, 22)
round1_plus_lm (1394, 26) (787, 26)


In [17]:
def evaluate_regression_both_scales(y_true_log, y_pred_log):
    # log scale metrics
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    r2_log = r2_score(y_true_log, y_pred_log)

    # original scale metrics
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "RMSE_log": rmse_log,
        "MAE_log": mae_log,
        "R2_log": r2_log,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

In [18]:
naive_pred = np.repeat(y_train.mean(), len(y_test))
naive_metrics = evaluate_regression_both_scales(y_test, naive_pred)

naive_results_df = pd.DataFrame([{
    "model": "NaiveMean",
    "dataset": "naive_mean",
    **naive_metrics
}])

naive_results_df

,model,dataset,RMSE_log,MAE_log,R2_log,RMSE,MAE,R2
0,NaiveMean,naive_mean,0.547993,0.444641,-0.25875,0.010087,0.006819,-0.035744


In [19]:
all_results = []

for dataset_name, cols in feature_sets.items():
    X_train = train_df[cols].copy()
    X_test = test_df[cols].copy()

    y_train = train_df[target_col].copy()
    y_test = test_df[target_col].copy()

    numeric_cols = [c for c in cols if c not in categorical_features]
    categorical_cols = categorical_features.copy()

    linear_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols)
        ]
    )

    tree_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols)
        ]
    )

    models = {
        "LinearRegression": Pipeline([
            ("prep", linear_preprocessor),
            ("model", LinearRegression())
        ]),
        "ElasticNet": Pipeline([
            ("prep", linear_preprocessor),
            ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42))
        ]),
        "RandomForest": Pipeline([
            ("prep", tree_preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=400,
                max_depth=10,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "XGBoost": Pipeline([
            ("prep", tree_preprocessor),
            ("model", XGBRegressor(
                n_estimators=400,
                max_depth=4,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_alpha=0.0,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            ))
        ])
    }

    for model_name, pipe in models.items():
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        metrics = evaluate_regression_both_scales(y_test, preds)
        metrics["model"] = model_name
        metrics["dataset"] = dataset_name
        all_results.append(metrics)

results_df = pd.DataFrame(all_results)
results_df = pd.concat([naive_results_df, results_df], ignore_index=True)
results_df = results_df.sort_values("RMSE").reset_index(drop=True)

results_df

,model,dataset,RMSE_log,MAE_log,R2_log,RMSE,MAE,R2
0,RandomForest,round1_baseline_price_context,0.425601,0.336016,0.240731,0.008750,0.005236,0.220710
1,RandomForest,round1_plus_lm,0.425090,0.334141,0.242553,0.008763,0.005198,0.218236
2,XGBoost,round1_plus_lm,0.436456,0.339010,0.201508,0.008890,0.005296,0.195412
3,XGBoost,round1_baseline_price_context,0.438021,0.342670,0.195773,0.008906,0.005351,0.192635
4,ElasticNet,round1_baseline_price_context,0.470734,0.371456,0.071159,0.009704,0.005845,0.041413
5,ElasticNet,round1_plus_lm,0.470008,0.368822,0.074024,0.009737,0.005810,0.034959
6,LinearRegression,round1_baseline_price_context,0.471936,0.372264,0.066409,0.009787,0.005872,0.024894
7,LinearRegression,round1_plus_lm,0.471125,0.369426,0.069617,0.009813,0.005836,0.019755
8,NaiveMean,naive_mean,0.547993,0.444641,-0.258750,0.010087,0.006819,-0.035744


In [20]:
results_df.to_csv("data/10_round1_improved_baseline/round1_model_results.csv", index=False)
print("Saved round 1 results.")

Saved round 1 results.


In [21]:
best_dataset = "round1_plus_lm"

numeric_cols = [c for c in feature_sets[best_dataset] if c not in categorical_features]
categorical_cols = categorical_features.copy()

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

rf_best = Pipeline([
    ("prep", tree_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

rf_best.fit(train_df[feature_sets[best_dataset]], y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['past_return_5d',
                                                   'past_return_10d',
                                                   'past_return_20d',
                                                   'abs_past_return_10d',
                                                   'past_realized_vol_5d',
                                                   'past_realized_vol_10d',
                                                   'past_realized_vol_20d',
                                                   'past_realized_vol_30d',
                                                   'past_realized_vol_60d',
                                                   'mean_abs_return_5d',
                                                   'mean_ab...
                                                   'log_text_length_words',
                                                   'lm_negative', 'lm_positive',
                                                   'lm_uncertainty',
                                                   'lm_net_sentiment']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['quarter', 'filing_month',
                                                   'filing_year'])])),
                ('model',
                 RandomForestRegressor(max_depth=10, min_samples_leaf=5,
                                       n_estimators=400, n_jobs=-1,
                                       random_state=42))])

In [22]:
prep = rf_best.named_steps["prep"]
model = rf_best.named_steps["model"]

feature_names = prep.get_feature_names_out()
importances = model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(20)

,feature,importance
10,num__mean_abs_return_10d,0.211274
8,num__past_realized_vol_60d,0.112960
7,num__past_realized_vol_30d,0.110098
6,num__past_realized_vol_20d,0.085635
9,num__mean_abs_return_5d,0.038944
0,num__past_return_5d,0.037375
28,cat__filing_month_2,0.032028
16,num__vol_ratio_10_60,0.029393
40,cat__filing_year_2020,0.028689
2,num__past_return_20d,0.027393


In [24]:
feat_imp.to_csv("data/10_round1_improved_baseline/round1_feature_importance.csv", index=False)
print("Saved round 1 feature importance.")

Saved round 1 feature importance.


In [25]:
from google.colab import files

files.download("data/10_round1_improved_baseline/round1_model_results.csv")
files.download("data/10_round1_improved_baseline/round1_feature_importance.csv")
files.download("data/10_round1_improved_baseline/round1_improved_baseline_dataset.csv")
files.download("data/10_round1_improved_baseline/round1_improved_baseline_dropped_rows.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>